In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import torch
import torch.nn.functional as F

from toy_image_generator import get_toy_image_example_batch
from diffuser_ddpm_linear_schedule import Diffuser_DDPM_linear_schedule
from unet import DiffusionUNet
from unet_example_config import DiffuserConfig

# Setup

In [ ]:
#Init
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(device)

torch.manual_seed(0)
np.random.seed(0)

In [ ]:
# Create scheduler model
ddpm_model = Diffuser_DDPM_linear_schedule(total_timesteps=1000, beta_start=0.0001, beta_end=0.02)

ddpm_model.betas = ddpm_model.betas.to(device)
ddpm_model.alphas = ddpm_model.alphas.to(device)
ddpm_model.alpha_bars = ddpm_model.alpha_bars.to(device)

In [ ]:
# Create model
config = DiffuserConfig()
unet = DiffusionUNet(
    config=config,
    model_in_channels=3,
    model_out_channels=3
).to(device)

optimizer = torch.optim.AdamW(unet.parameters(), lr=2e-4)
scaler = torch.GradScaler("cuda") if device.type == "cuda" else None

In [ ]:
# Dataloader
def make_toy_batch(batch_size=8, x_dim=64, y_dim=64):
    hue_1s = np.random.randint(0, 180, size=batch_size)
    hue_2s = np.random.randint(0, 180, size=batch_size)
    hue_3s = np.zeros(batch_size)
    saturation_1s = np.full(batch_size, 255)
    saturation_2s = np.full(batch_size, 255)
    saturation_3s = np.zeros(batch_size)
    value_1s = np.full(batch_size, 255)
    value_2s = np.full(batch_size, 255)
    value_3s = np.zeros(batch_size)
    angles = np.random.randint(15, 75, size=batch_size)
    line_widths = np.random.randint(3, 9, size=batch_size)

    batch = get_toy_image_example_batch(
        x_dim=x_dim,
        y_dim=y_dim,
        hue_1s=hue_1s,
        saturation_1s=saturation_1s,
        value_1s=value_1s,
        hue_2s=hue_2s,
        saturation_2s=saturation_2s,
        value_2s=value_2s,
        hue_3s=hue_3s,
        saturation_3s=saturation_3s,
        value_3s=value_3s,
        angles=angles,
        line_widths=line_widths,
    )
    return batch.to(device)

In [ ]:
def train_step(data, model, ddpm):
    model.train()
    optimizer.zero_grad(set_to_none=True)

    B = data.shape[0]
    t = torch.randint(0, ddpm.total_timesteps, (B,), device=device)

    x_t, true_noise = ddpm.forward_diffusion(data, t)

    if device.type == "cuda":
        with torch.autocast("cuda", dtype=torch.float16):
            pred_noise = model(x_t, t)
            loss = F.mse_loss(pred_noise, true_noise)
        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()
    else:
        pred_noise = model(x_t, t)
        loss = F.mse_loss(pred_noise, true_noise)
        loss.backward()
        optimizer.step()

    return loss.item()

# Sampling

In [ ]:
@torch.no_grad()
def sample(model, ddpm, shape, fixed_noise=None):
    model.eval()

    x = torch.randn(shape, device=device)

    if fixed_noise != None:
        x = fixed_noise

    for t in range(ddpm.total_timesteps - 1, 0, -1):
        ts = torch.full((shape[0],), t, device=device, dtype=torch.long)
        pred_noise = model(x, ts)
        x = ddpm.reverse_diffusion(x, ts, pred_noise)

    return x

In [ ]:
def show_images(img_batch, title=None):
    img_batch = img_batch.detach().cpu()
    fig, axes = plt.subplots(1, img_batch.shape[0], figsize=(4 * img_batch.shape[0], 4))
    if img_batch.shape[0] == 1:
        axes = [axes]

    for i, ax in enumerate(axes):
        img = img_batch[i]
        img = (img + 1.0) / 2.0
        img = (img * 255.0).clamp(0, 255).byte()
        img = img.permute(1, 2, 0).numpy()
        ax.imshow(img)
        ax.axis("off")

    if title:
        fig.suptitle(title)
    plt.show()

In [ ]:
@torch.no_grad()
def visualize_reverse_process(model, ddpm, x0):
    model.eval()
    ts = torch.full((x0.shape[0],), ddpm.total_timesteps - 1, device=device, dtype=torch.long)
    x_t, _ = ddpm.forward_diffusion(x0, ts)

    snapshots = {}
    visualize_steps = [999, 750, 500, 250, 1]

    for t in range(ddpm.total_timesteps - 1, 0, -1):
        ts = torch.full((x_t.shape[0],), t, device=device, dtype=torch.long)
        pred_noise = model(x_t, ts)
        x_t = ddpm.reverse_diffusion(x_t, ts, pred_noise)

        if t in visualize_steps:
            snapshots[t] = x_t.clone()

    return snapshots

# Run and Show

In [ ]:
# training loop:
num_steps = 5000
batch_size = 8
loss_history = []

fixed_noise = torch.randn(4, 3, 64, 64, device=device)
preview_every = 200

for step in range(num_steps):
    data = make_toy_batch(batch_size=batch_size, x_dim=64, y_dim=64)
    loss = train_step(data, unet, ddpm_model)
    loss_history.append(loss)

    if step % 100 == 0:
        print(f"step {step:5d} | loss {loss:.4f}")

    if step % preview_every == 0:
        unet.eval()
        with torch.no_grad():
            samples = sample(unet, ddpm_model, fixed_noise.shape, fixed_noise=fixed_noise)
        show_images(samples, title=f"step {step}")
        unet.train()

In [ ]:
show_images(samples, title="Generated samples")

In [ ]:
samples = sample(unet, ddpm_model, (4, 3, 64, 64))